# ScrollGT quickstart

Score an ink prediction against **registered human ground truth** on the open Vesuvius SOTA data.

This notebook: installs ScrollGT, loads the held-out flagship target, and scores two reference predictions — a constant map (to show the anti-gaming gate) and the published canon-teacher region if you supply one.

In [ ]:
!git clone --depth 1 https://github.com/jonmarrs/scrollgt.git
%pip install -q -e scrollgt/

In [ ]:
import numpy as np
from scrollgt import score_prediction, load_target
from scrollgt.metrics import segmentation_metrics

TARGET = "scrollgt/data/scroll1_20231210121321"  # held-out flagship
gt, mask, meta = load_target(TARGET)
print("target:", meta["target_id"], "| gt shape:", gt.shape, "| ink prevalence:", round(float(gt.mean()), 4))
print("predict-region spec:", meta["region"])

## The anti-gaming demonstration

A constant all-positive prediction gets the trivial F1 (2p/(1+p)) *for free* — but `ap_prevalence_lift` stays ≈ 1.0 and ROC-AUC = 0.5. That is why ScrollGT's headline is F1 **gated by lift**, and why near-chance results in `baselines/BASELINES.md` are trustworthy.

In [ ]:
constant = np.full(gt.shape, 0.9)
m = segmentation_metrics(constant, gt, mask)
p = m["positive_rate"]
print(f"constant predictor: val_f1={m['val_f1']:.4f} (trivial 2p/(1+p)={2*p/(1+p):.4f})")
print(f"                    ap_prevalence_lift={m['ap_prevalence_lift']:.4f}  roc_auc={m['roc_auc']:.4f}")

## Score your own model

Predict a probability map over **exactly** the region in `meta["region"]` (SOTA S3 zarr, pyramid level 2, y0/x0/size, 26 mid-depth slices), save as 8-bit PNG or `.npy` in [0,1], then:

In [ ]:
# result = score_prediction("my_prediction.png", TARGET)
# print(result["metrics"]["val_f1"], result["metrics"]["ap_prevalence_lift"], result["metrics"]["roc_auc"])
print("Baselines to beat (held-out): every published model sits at roc_auc ≈ 0.50–0.56.")
print("Beating ROC 0.60 held-out, honestly, would be news — see baselines/BASELINES.md.")

## Column-level target (v0.2): PHerc 1667 merged geometry

The first non-training-scroll target has **no pixel GT** — scoring measures
*consistency with the published reading* (22 scholar-validated columns) at column
granularity. Floor: constant and papyrus-mask predictions score exactly 0.5.


In [ ]:
import numpy as np

# demo on a partial extent: paint the two easiest text columns from the public layout
# (this is the disclosed geometry-oracle 'cheat' — it proves layout-consistency only,
#  never reading; real submissions must include their prediction map for review)
import json
cols = json.load(open('../data/pherc1667_merged_columns/columns.json'))['columns']
c17, c18 = [c for c in cols if c['col'] in (17, 18)]
y0, x0 = 100, c17['gx0'] - 50
pred = np.zeros((1850, c18['gx1'] + 50 - x0), dtype=np.float32)
for c in (c17, c18):
    (b0, b1) = c['text_band']
    pred[b0 - y0:b1 - y0, c['gx0'] - x0:c['gx1'] - x0] = 1.0
np.save('oracle_demo.npy', pred)
!cd .. && scrollgt score-columns notebooks/oracle_demo.npy data/pherc1667_merged_columns --origin 100 {x0}
